In [1]:
from getpass import getpass
import os

# Prompt for API key (hidden input)
api_key = getpass("Enter your Gemini API key: ")

# Set as environment variable
os.environ["GOOGLE_API_KEY"] = api_key

print("✅ Key loaded successfully!")

✅ Key loaded successfully!


In [2]:
!pip install -q langchain langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.7 MB/s eta 0:00:00


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

Tool creation

In [4]:
@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [5]:
print(multiply.invoke({'a':3, 'b':4}))

12


Tool Binding

In [6]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  # or "gemini-1.5-flash" for faster/cheaper
    temperature=0
)

Tool Calling

In [7]:
llm.invoke('hi')

AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ee0e2-71b3-70c2-a3f8-aa15d67fe886-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 243, 'total_tokens': 245, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 233}})

In [8]:
llm_with_tools = llm.bind_tools([multiply])

In [9]:
query = HumanMessage('can you multiply 3 with 1000')

In [10]:
messages = [query]

In [11]:
result = llm_with_tools.invoke(messages)

In [12]:
messages.append(result)

Tool Execution

In [13]:
tool_result = multiply.invoke(result.tool_calls[0])

In [14]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='7d8264d3-837b-45cd-89f2-77e807e99f06')

In [15]:
messages.append(tool_result)

In [16]:
llm_with_tools.invoke(messages).content

'The product of 3 and 1000 is 3000.'

Real Time Currency Convertor

In [17]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/cc2fee393e72eaad68b45265/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate

In [18]:
get_conversion_factor.args

{'base_currency': {'title': 'Base Currency', 'type': 'string'},
 'target_currency': {'title': 'Target Currency', 'type': 'string'}}

In [19]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [20]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1781827202,
 'time_last_update_utc': 'Fri, 19 Jun 2026 00:00:02 +0000',
 'time_next_update_unix': 1781913602,
 'time_next_update_utc': 'Sat, 20 Jun 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.4149}

In [21]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0
)

In [23]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [24]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [25]:
ai_message = llm_with_tools.invoke(messages)

In [26]:
messages.append(ai_message)

In [27]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

In [28]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"target_currency": "USD", "base_currency": "INR"}'}, '__gemini_function_call_thought_signatures__': {'06a5008d-737e-498f-90e7-f20f44127a4d': 'CuYGAQw51sdjlS9qflEEBu0pc3f+EnjUaRBOM7K/CcsmnfosFoa2vFFkh6WTuWujF4tYBonCVknx3RSe9pgfHtpaAJxsRJPo5h/Znp11YLQa50dhkIjy4hX3FgNQ8+p2ZSNU862HnXOh6zh3dkP8luKaHUArWoBktG6zcYT+nyAr3lVxkBy7ySFLFdWRdE7G/jurH4sskw2TTq6jUkeIetf1/b5fJ4WacAck8YVPSaC0ZIdh4rZtQo/LFwuBwedEdpuJoDUXJaHPf6sGFpgHKy/l3GM0AltF6IqNLgKrT6BC8tQQIXu/Afe5xP5ZRTpNTyY/zNUcsa2AolWHok7f7+VI9tOw93dot99tfvmJ3/fR5IQGZ6eQ9aShvSN700p1ERvYaQO7AzDQTHbzzmey1vzyCcCmWszNbu9oZjbwu1ziSiBLcHFn7rlJ/JxJd7HnUwm+WSGsfI1wwa3I0iiYa6dgYKtx7zx24nFTvmcRk+SkKx0E2RzdlJAd21R0MUfhuAELi1SUMumdS5bMd1sG9pQrMPnr8stGg7XkCQqsEsweywZb7rdCzfJEAoCxNtOrmQCYk

In [29]:
llm_with_tools.invoke(messages).content

''

In [32]:
from langgraph.prebuilt import create_react_agent
agent = create_react_agent(llm, [get_conversion_factor, convert])

# Run
response = agent.invoke({
    "messages": [("human", "What is 10 INR in USD?")]
})

print(response["messages"][-1].content)

/tmp/ipykernel_2927/2986436919.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, [get_conversion_factor, convert])


[{'type': 'text', 'text': 'I can only provide the conversion factor between INR and USD. Would you like to know that?', 'extras': {'signature': 'CukVAQw51scSR8Pfz7bAKrHuRA7iGudGP4aolXWWo/V+u9ALfCsIzPRxnowK7mU6deDHswrfmZ1LtnMXdtQCqjTdg5c0pwkue8ISepae0z8LP+icG4lzJb43j0gXu3SX/mgEaFd26chkENF8iyBJEsXHUBUtz8Y0ANjinmOJTpgFOw8X5piBFUvYIysm6TybVkI11MUqNtrtWCeftW+/heZE0fL3OtOfBVMSHeBxJD1gzg0fP/6o9vC19/DEswiA9d5v+c2pFYjGJzSo5rghLxd+dg1b2xyNcR9lBhLQW8DflhXEpVB+4npia1Dy+IyXrWZYKvsG2WPoR0tNX2js5bdjKLUqhEBqojvYL+tEbkf2dh1RYa0+1EMpsSHXTtMwQnzD418wtlRejVJXRS+XPucaPUSiemIvHbWqTre/ypTnGb04rkMVpp37PbPI4APVEu+FaLBX0WFTWh3Lpml20Gl8IKmGwXMxetwaE6pHCnmVXwSlKSI4b+h7TTQx7mMGBnqmauFsHan1Fmelf8442jKiWKM8qmrdAFBHg/GY3sxnp9a9bBQXRUEV+FC9/VO3eyIAauL+oZmNaho1/qK2LQMljExYnGWC+osAbjPf+cR9DzgwXTpWTXGxb2vdlym317Zj357Fe8eY0F0XHKEYLSswDxP/omKfpENHxmnauGK1PfeuLjXW9Qz6elmA80pJb/qm+cYhmZGbg2N20XUq9I/knGsPT3paBkj7TTmwOcM7zZ3+rDG5kRydolaO4y9Wk0SWHJfbRL0IruT0oair6LEP5G8gHJpS6J05EEkLmkk44tcrxcnOb+2whLxdc6DDELMdeCllrUujhp3D2cuJaTp